In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/cs-training.csv", index_col=0)

# Diagnóstico profesional de calidad del dato
print("=" * 55)
print("  REPORTE DE CALIDAD DE DATOS")
print("=" * 55)

total_filas = len(df)

for col in df.columns:
    nulos    = df[col].isnull().sum()
    pct_nulo = nulos / total_filas * 100
    minimo   = df[col].min()
    maximo   = df[col].max()
    alerta   = "\u26A0\uFE0F" if pct_nulo > 5 else ""
    print(f"{col:<50} nulos: {pct_nulo:5.1f}%{alerta}|  min: {minimo}  max: {maximo}")

  REPORTE DE CALIDAD DE DATOS
SeriousDlqin2yrs                                   nulos:   0.0%|  min: 0  max: 1
RevolvingUtilizationOfUnsecuredLines               nulos:   0.0%|  min: 0.0  max: 50708.0
age                                                nulos:   0.0%|  min: 0  max: 109
NumberOfTime30-59DaysPastDueNotWorse               nulos:   0.0%|  min: 0  max: 98
DebtRatio                                          nulos:   0.0%|  min: 0.0  max: 329664.0
MonthlyIncome                                      nulos:  19.8%⚠️|  min: 0.0  max: 3008750.0
NumberOfOpenCreditLinesAndLoans                    nulos:   0.0%|  min: 0  max: 58
NumberOfTimes90DaysLate                            nulos:   0.0%|  min: 0  max: 98
NumberRealEstateLoansOrLines                       nulos:   0.0%|  min: 0  max: 54
NumberOfTime60-89DaysPastDueNotWorse               nulos:   0.0%|  min: 0  max: 98
NumberOfDependents                                 nulos:   2.6%|  min: 0.0  max: 20.0


In [27]:
# MonthlyIncome: imputamos con la MEDIANA (no la media, porque el ingreso
# tiene distribución sesgada a la derecha — los outliers inflan la media)
mediana_ingreso = df["MonthlyIncome"].median()
df["MonthlyIncome"].fillna(mediana_ingreso, inplace=True)
print(f"✅ MonthlyIncome: nulos imputados con mediana = ${mediana_ingreso:,.0f}")

# NumberOfDependents: imputamos con 0 (criterio conservador:
# si no declaró dependientes, asumimos que no tiene)
df["NumberOfDependents"].fillna(0, inplace=True)
print(f"✅ NumberOfDependents: nulos imputados con 0")

# Verificación
nulos_restantes = df.isnull().sum().sum()
print(f"\n✅ Nulos restantes en el dataset: {nulos_restantes}")

✅ MonthlyIncome: nulos imputados con mediana = $5,400
✅ NumberOfDependents: nulos imputados con 0

✅ Nulos restantes en el dataset: 0


C:\Users\kenpa\AppData\Local\Temp\ipykernel_29992\2610192471.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["MonthlyIncome"].fillna(mediana_ingreso, inplace=True)
C:\Users\kenpa\AppData\Local\Temp\ipykernel_29992\2610192471.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

In [ ]:
# Estos valores extremos existen en el dataset real y se deben tratar
print("=== VALORES SOSPECHOSOS ===\n")

# Edad: hay registros con age == 0, imposible en un cliente bancario
print(f"Clientes con age == 0: {(df['age'] == 0).sum()}")
df = df[df["age"] > 0]
print(f"✅ Eliminados. Filas restantes: {len(df):,}")

# Utilización de crédito rotativo: valores > 1 indican sobreendeudamiento
# valores absurdamente altos son errores de captura
pct_alto = (df["RevolvingUtilizationOfUnsecuredLines"] > 1).sum()
print(f"\nClientes con utilización > 100%: {pct_alto:,} ({pct_alto/len(df)*100:.1f}%)")

# Cappear en 1.5 (máximo razonable con tolerancia)
df["RevolvingUtilizationOfUnsecuredLines"] = df[
    "RevolvingUtilizationOfUnsecuredLines"
].clip(upper=1.5)
print("✅ Utilización cappeada en 1.5")

# DebtRatio: misma lógica — valores absurdos son errores
p99_debt = df["DebtRatio"].quantile(0.99)
df["DebtRatio"] = df["DebtRatio"].clip(upper=p99_debt)
print(f"\n✅ DebtRatio cappeado en percentil 99 = {p99_debt:,.1f}")

# MonthlyIncome: cappear en percentil 99 también
p99_income = df["MonthlyIncome"].quantile(0.99)
df["MonthlyIncome"] = df["MonthlyIncome"].clip(upper=p99_income)
print(f"✅ MonthlyIncome cappeado en percentil 99 = ${p99_income:,.0f}")

=== VALORES SOSPECHOSOS ===

Clientes con age == 0: 1
✅ Eliminados. Filas restantes: 149,999

Clientes con utilización > 100%: 3,321 (2.2%)
✅ Utilización cappeada en 1.5

✅ DebtRatio cappeado en percentil 99 = 4,979.1
✅ MonthlyIncome cappeado en percentil 99 = $23,000


In [ ]:
# Nombres limpios, en español, sin caracteres especiales

df.rename(columns={
    "SeriousDlqin2yrs":                          "mora_grave",
    "RevolvingUtilizationOfUnsecuredLines":       "uso_credito_rotativo",
    "age":                                        "edad",
    "NumberOfTime30-59DaysPastDueNotWorse":       "atrasos_30_59d",
    "DebtRatio":                                  "ratio_deuda",
    "MonthlyIncome":                              "ingreso_mensual",
    "NumberOfOpenCreditLinesAndLoans":            "lineas_credito_abiertas",
    "NumberOfTimes90DaysLate":                    "atrasos_90d",
    "NumberRealEstateLoansOrLines":               "prestamos_inmobiliarios",
    "NumberOfTime60-89DaysPastDueNotWorse":       "atrasos_60_89d",
    "NumberOfDependents":                         "dependientes"
}, inplace=True)

print("✅ Columnas renombradas:")
for col in df.columns:
    print(f"   · {col}")

✅ Columnas renombradas:
   · mora_grave
   · uso_credito_rotativo
   · edad
   · atrasos_30_59d
   · ratio_deuda
   · ingreso_mensual
   · lineas_credito_abiertas
   · atrasos_90d
   · prestamos_inmobiliarios
   · atrasos_60_89d
   · dependientes


In [ ]:
# 1. Total de atrasos históricos (ventanas de tiempo)
df["total_atrasos"] = (
    df["atrasos_30_59d"] +
    df["atrasos_60_89d"] +
    df["atrasos_90d"]
)

# 2. Segmento de riesgo basado en atrasos (lógica scoring bancario)
def segmentar_riesgo(row):
    if row["atrasos_90d"] >= 1 or row["total_atrasos"] >= 3:
        return "Alto"
    elif row["total_atrasos"] >= 1 or row["uso_credito_rotativo"] > 0.75:
        return "Medio"
    else:
        return "Bajo"

df["segmento_riesgo"] = df.apply(segmentar_riesgo, axis=1)

# 3. Ingreso por dependiente (capacidad de pago real)
df["ingreso_por_dependiente"] = df["ingreso_mensual"] / (df["dependientes"] + 1)

# Distribución de segmentos
print("=== DISTRIBUCIÓN DE SEGMENTO DE RIESGO ===\n")
dist = df["segmento_riesgo"].value_counts()
for seg, cnt in dist.items():
    pct = cnt / len(df) * 100
    barra = "█" * int(pct / 2)
    print(f"  {seg:<8} {barra:<30} {cnt:>7,}  ({pct:.1f}%)")

=== DISTRIBUCIÓN DE SEGMENTO DE RIESGO ===

  Bajo     ██████████████████████████████████ 104,125  (69.4%)
  Medio    ███████████                     34,581  (23.1%)
  Alto     ███                             11,293  (7.5%)


In [ ]:
# Guardamos en dos formatos: CSV para Power BI, y también SQLite
output_path = "../data/processed/credito_limpio.csv"
df.to_csv(output_path, index=False)

print(f"✅ Dataset limpio guardado en: {output_path}")
print(f"   Filas finales:   {len(df):,}")
print(f"   Columnas finales: {len(df.columns)}")
print(f"   Columnas: {list(df.columns)}")

✅ Dataset limpio guardado en: ../data/processed/credito_limpio.csv
   Filas finales:   149,999
   Columnas finales: 14
   Columnas: ['mora_grave', 'uso_credito_rotativo', 'edad', 'atrasos_30_59d', 'ratio_deuda', 'ingreso_mensual', 'lineas_credito_abiertas', 'atrasos_90d', 'prestamos_inmobiliarios', 'atrasos_60_89d', 'dependientes', 'total_atrasos', 'segmento_riesgo', 'ingreso_por_dependiente']


In [32]:
import sqlalchemy as sa

# Crear base de datos SQLite local
engine = sa.create_engine("sqlite:///../data/processed/credito.db")

# Cargar el DataFrame como tabla SQL
df.to_sql("clientes", con=engine, if_exists="replace", index=False)
print("✅ Datos cargados en SQLite como tabla 'clientes'")

# Consulta 1: tasa de mora por segmento de riesgo
query_mora = """
SELECT
    segmento_riesgo,
    COUNT(*)                                      AS total_clientes,
    SUM(mora_grave)                               AS clientes_en_mora,
    ROUND(AVG(mora_grave) * 100, 2)               AS tasa_mora_pct,
    ROUND(AVG(ingreso_mensual), 0)                AS ingreso_promedio,
    ROUND(AVG(uso_credito_rotativo) * 100, 1)     AS uso_credito_pct
FROM clientes
GROUP BY segmento_riesgo
ORDER BY tasa_mora_pct DESC
"""

resultado = pd.read_sql(query_mora, engine)
print("\n=== TASA DE MORA POR SEGMENTO ===")
display(resultado)

✅ Datos cargados en SQLite como tabla 'clientes'

=== TASA DE MORA POR SEGMENTO ===


,segmento_riesgo,total_clientes,clientes_en_mora,tasa_mora_pct,ingreso_promedio,uso_credito_pct
0,Alto,11293,4412,39.07,5369.0,72.3
1,Medio,34581,3656,10.57,5886.0,67.2
2,Bajo,104125,1958,1.88,6311.0,16.3
